# BNEPA Royalty Report Analysis

Source: `BNEPA_Royalty_Report_MAY2026_LV.xlsx` — monthly China royalty statements.

**Task 1:** Extract the distinct list of `Product Name` + `Item Number` across all monthly sheets.

In [1]:
import pandas as pd
from pathlib import Path

WORKBOOK = Path("BNEPA_Royalty_Report_MAY2026_LV.xlsx")
SKIP_SHEETS = {"New template", "銷售庫存統計總表"}
HEADER_SCAN_DEPTH = 12  # rows scanned for the 'Product Name' header


def normalize_name(name: object) -> str:
    """Canonical key for a product name.

    Strips trademark / registered / copyright marks first, then drops every
    non-alphanumeric code point and uppercases. NaN / None / non-str → "".
    Single source of truth — reused by Task 1 (distinct products) and the
    SRP / promo histories.
    """
    if not isinstance(name, str):
        return ""
    s = name.replace("™", "").replace("®", "").replace("©", "")
    return "".join(c for c in s if c.isalnum()).upper()

In [2]:
def find_header_row(path: Path, sheet: str, depth: int = HEADER_SCAN_DEPTH) -> int:
    """Return the 0-indexed row that contains the 'Product Name' header."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    for i in range(len(probe)):
        cells = {str(v).strip() for v in probe.iloc[i].tolist() if pd.notna(v)}
        if "Product Name" in cells:
            return i
    raise ValueError(f"No 'Product Name' header found in first {depth} rows of {sheet!r}")

xl = pd.ExcelFile(WORKBOOK, engine="openpyxl")
monthly = [s for s in xl.sheet_names if s not in SKIP_SHEETS]
frames = {}
for s in monthly:
    hdr = find_header_row(WORKBOOK, s)
    frames[s] = pd.read_excel(xl, sheet_name=s, header=hdr, engine="openpyxl")
    print(f"  {s}: header at row {hdr + 1}, {len(frames[s])} data rows")

print(f"\nLoaded {len(frames)} monthly sheets")

  8月9月: header at row 8, 120 data rows


  10月: header at row 8, 123 data rows


  11月: header at row 8, 234 data rows


  12月: header at row 8, 203 data rows


  1月: header at row 8, 177 data rows


...

  12月2025: header at row 7, 463 data rows


  1月2026: header at row 7, 446 data rows


  2月2026 : header at row 7, 508 data rows


  3月2026: header at row 7, 476 data rows


  4月2026: header at row 7, 450 data rows

Loaded 20 monthly sheets


In [3]:
parts = []
for name, df in frames.items():
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    if not pn or not inum:
        print(f"  [skip] {name}: missing columns ({list(df.columns)[:5]}…)")
        continue
    sub = df[[pn, inum]].rename(columns={pn: "Product Name", inum: "Item Number"})
    sub["sheet"] = name
    parts.append(sub)

combined = pd.concat(parts, ignore_index=True)

combined = combined.dropna(subset=["Product Name", "Item Number"], how="any")
combined["Product Name"] = combined["Product Name"].astype(str).str.strip()
combined["Item Number"] = combined["Item Number"].astype(str).str.strip()
combined = combined[(combined["Product Name"] != "") & (combined["Item Number"] != "")]
combined["Normalized Name"] = combined["Product Name"].map(normalize_name)
combined = combined[combined["Normalized Name"] != ""]

# Workbook stores monthly sheets in chronological order, so position in
# `monthly` doubles as a recency rank. Keep one row per Normalized Name —
# the variant from the most recent sheet that carried it.
sheet_rank = {s: i for i, s in enumerate(monthly)}
combined["_rank"] = combined["sheet"].map(sheet_rank)

distinct = (
    combined.sort_values("_rank")
    .drop_duplicates(subset=["Normalized Name"], keep="last")[
        ["Product Name", "Item Number", "Normalized Name"]
    ]
    .sort_values(["Product Name", "Item Number"])
    .reset_index(drop=True)
)
combined = combined.drop(columns="_rank")

assert distinct["Normalized Name"].is_unique, "Normalized Name column is not unique"
print(f"{len(distinct)} distinct products (by Normalized Name)")
distinct

185 distinct products (by Normalized Name)


,Product Name,Item Number,Normalized Name
0,.hack//G.U. Last Recode,E05367,HACKGULASTRECODE
1,ACE COMBAT 7: SKIES UNKNOWN,E03174,ACECOMBAT7SKIESUNKNOWN
2,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKEDITION
3,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05444,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKULTIMATEED...
4,ARMORED CORE VI FIRES OF RUBICON,E05837,ARMOREDCOREVIFIRESOFRUBICON
...,...,...,...
180,That Time I Got Reincarnated as a Slime ISEKAI...,E06534,THATTIMEIGOTREINCARNATEDASASLIMEISEKAICHRONICL...
181,Towa and the Guardians of the Sacred Tree,E06936,TOWAANDTHEGUARDIANSOFTHESACREDTREE
182,Towa and the Guardians of the Sacred Tree Delu...,L00117,TOWAANDTHEGUARDIANSOFTHESACREDTREEDELUXEEDITION
183,We Love Katamari REROLL+ Royal Reverie,E05071,WELOVEKATAMARIREROLLROYALREVERIE


In [4]:
out = Path("distinct_products.csv")
distinct.to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/distinct_products.csv


## Task 2 — SRP history per product

For each (Product Name, Item Number), list the SRP (CNY) values it had and the month range each one applied to. Consecutive months at the same SRP collapse into a single range; a gap in sheets (a month where the product didn't sell) breaks the run.

In [5]:
import re
import datetime as _dt

DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")

def find_sales_period(path: Path, sheet: str, depth: int = 12) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Return (start_date, end_date) for the sheet by scanning rows below 'Sales Period :'."""
    probe = pd.read_excel(path, sheet_name=sheet, header=None, nrows=depth, engine="openpyxl")
    dates: list[pd.Timestamp] = []
    started = False
    for i in range(len(probe)):
        for v in probe.iloc[i].tolist():
            if pd.isna(v):
                continue
            if isinstance(v, str) and "Sales Period" in v:
                started = True
                continue
            if not started:
                continue
            if isinstance(v, (pd.Timestamp, _dt.datetime, _dt.date)):
                dates.append(pd.Timestamp(v))
            elif isinstance(v, str) and DATE_RE.match(v):
                dates.append(pd.Timestamp(v))
            if len(dates) == 2:
                return dates[0], dates[1]
    raise ValueError(f"Could not find sales-period dates in {sheet!r}")

sheet_periods = {s: find_sales_period(WORKBOOK, s) for s in monthly}
sheet_order = sorted(sheet_periods, key=lambda s: sheet_periods[s][0])
for s in sheet_order:
    a, b = sheet_periods[s]
    print(f"  {s}: {a.date()} → {b.date()}")

  8月9月: 2024-08-01 → 2024-09-30
  10月: 2024-10-01 → 2024-10-31
  11月: 2024-11-01 → 2024-11-30
  12月: 2024-12-01 → 2024-12-31
  1月: 2025-01-01 → 2025-01-31
  2月: 2025-02-01 → 2025-02-28
  3月: 2025-03-01 → 2025-03-31
  4月: 2025-04-01 → 2025-04-30
  5月: 2025-05-01 → 2025-05-31
  6月: 2025-06-01 → 2025-06-30
  7月: 2025-07-01 → 2025-07-31
  8月: 2025-08-01 → 2025-08-30
  9月: 2025-09-01 → 2025-09-30
  10月2025: 2025-10-01 → 2025-10-31
  11月2025: 2025-11-01 → 2025-11-30
  12月2025: 2025-12-01 → 2025-12-31
  1月2026: 2026-01-01 → 2026-01-31
  2月2026 : 2026-02-01 → 2026-02-28
  3月2026: 2026-03-01 → 2026-03-31
  4月2026: 2026-04-01 → 2026-04-30


In [6]:
srp_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn, inum, srp = cols.get("Product Name"), cols.get("Item Number"), cols.get("SRP (CNY)")
    if not (pn and inum and srp):
        print(f"  [skip] {sheet}: missing one of Product Name / Item Number / SRP (CNY)")
        continue
    sub = df[[pn, inum, srp]].rename(
        columns={pn: "Product Name", inum: "Item Number", srp: "SRP (CNY)"}
    )
    sub = sub.dropna(subset=["Product Name", "Item Number", "SRP (CNY)"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub["Normalized Name"] = sub["Product Name"].map(normalize_name)
    sub = sub[sub["Normalized Name"] != ""]
    sub["SRP (CNY)"] = pd.to_numeric(sub["SRP (CNY)"], errors="coerce")
    sub = sub.dropna(subset=["SRP (CNY)"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    srp_parts.append(sub)

srp_long = pd.concat(srp_parts, ignore_index=True)

# Within a single sheet a normalized product can appear on multiple rows
# (different Customer values or trivially different name variants). Keep the
# modal SRP and the display Product Name from the alphabetically-first variant.
srp_long = (
    srp_long.groupby(
        ["Normalized Name", "sheet", "period_start", "period_end"], as_index=False
    ).agg(
        **{
            "Product Name": ("Product Name", "first"),
            "SRP (CNY)": ("SRP (CNY)", lambda s: s.mode().iloc[0]),
        }
    )
)
print(f"{len(srp_long)} (normalized product, sheet) SRP observations")
srp_long.head()

2799 (normalized product, sheet) SRP observations


,Normalized Name,sheet,period_start,period_end,Product Name,SRP (CNY)
0,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,10月,2024-10-01,2024-10-31,Accel World VS. Sword Art Online Deluxe Edition,278.0
1,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,10月2025,2025-10-01,2025-10-31,Accel World VS. Sword Art Online Deluxe Edition,278.0
2,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,11月,2024-11-01,2024-11-30,Accel World VS. Sword Art Online Deluxe Edition,278.0
3,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,11月2025,2025-11-01,2025-11-30,Accel World VS. Sword Art Online Deluxe Edition,278.0
4,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,12月,2024-12-01,2024-12-31,Accel World VS. Sword Art Online Deluxe Edition,278.0


In [7]:
# Sort each product's observations chronologically, then collapse consecutive
# same-SRP rows into ranges. A "gap" (sheet missing between two observations)
# breaks the run.
sheet_idx = {s: i for i, s in enumerate(sheet_order)}
srp_long["_idx"] = srp_long["sheet"].map(sheet_idx)
srp_long = srp_long.sort_values(["Normalized Name", "_idx"]).reset_index(drop=True)

runs = []
for norm, grp in srp_long.groupby("Normalized Name", sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    run_srp = None
    prev_idx = None
    for _, row in grp.iterrows():
        if run_srp is None:
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        if row["SRP (CNY)"] == run_srp and row["_idx"] == prev_idx + 1:
            run_end = row["period_end"]
        else:
            runs.append((norm, run_srp, run_start, run_end))
            run_srp = row["SRP (CNY)"]
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if run_srp is not None:
        runs.append((norm, run_srp, run_start, run_end))

srp_history = pd.DataFrame(
    runs, columns=["Normalized Name", "SRP (CNY)", "start_date", "end_date"]
)

# Attach the display Product Name from the most recent sheet that carried
# the normalized product, using the same `_rank` map built in Task 1.
display_names = (
    combined.assign(_rank=combined["sheet"].map(sheet_rank))
    .sort_values("_rank")
    .drop_duplicates(subset=["Normalized Name"], keep="last")
    .set_index("Normalized Name")["Product Name"]
)
srp_history.insert(0, "Product Name", srp_history["Normalized Name"].map(display_names))
srp_history["start_month"] = srp_history["start_date"].dt.strftime("%Y-%m")
srp_history["end_month"] = srp_history["end_date"].dt.strftime("%Y-%m")
srp_history = srp_history[
    ["Product Name", "Normalized Name", "SRP (CNY)", "start_month", "end_month",
     "start_date", "end_date"]
].sort_values(["Product Name", "start_date"]).reset_index(drop=True)

print(f"{len(srp_history)} SRP runs across {srp_history['Normalized Name'].nunique()} products")
srp_history.head(20)

342 SRP runs across 185 products


,Product Name,Normalized Name,SRP (CNY),start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,HACKGULASTRECODE,278.0,2024-08,2025-04,2024-08-01,2025-04-30
1,.hack//G.U. Last Recode,HACKGULASTRECODE,278.0,2025-06,2026-04,2025-06-01,2026-04-30
2,ACE COMBAT 7: SKIES UNKNOWN,ACECOMBAT7SKIESUNKNOWN,268.0,2024-08,2026-04,2024-08-01,2026-04-30
3,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKEDITION,368.0,2024-08,2024-10,2024-08-01,2024-10-31
4,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKEDITION,368.0,2025-01,2026-04,2025-01-01,2026-04-30
5,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKULTIMATEED...,498.0,2024-08,2024-10,2024-08-01,2024-10-31
6,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,ACECOMBAT7SKIESUNKNOWNTOPGUNMAVERICKULTIMATEED...,498.0,2025-01,2026-04,2025-01-01,2026-04-30
7,ARMORED CORE VI FIRES OF RUBICON,ARMOREDCOREVIFIRESOFRUBICON,298.0,2024-08,2024-09,2024-08-01,2024-09-30
8,ARMORED CORE VI FIRES OF RUBICON,ARMOREDCOREVIFIRESOFRUBICON,298.0,2025-12,2025-12,2025-12-01,2025-12-31
9,ARMORED CORE VI FIRES OF RUBICON Deluxe Edition,ARMOREDCOREVIFIRESOFRUBICONDELUXEEDITION,348.0,2024-08,2026-04,2024-08-01,2026-04-30


In [8]:
out = Path("product_srp_history.csv")
srp_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")

Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_srp_history.csv


## Task 3 — Promotion history per (product, customer)

For each (Product Name, Item Number, Customer, Promo Discount), list the month range each promo was active. Customer is from the newer-layout sheets; the older sheets (8月9月 → 6月) have no Customer column and are labelled `All`. Rows where `Promo Discount = 0` (no promo) and aggregate rows (`SUBTOTAL`, `TOTAL`) are excluded.

In [9]:
PROMO_COL_VARIANTS = ("Promo Discount\n(OFF)", "Promo Discount (OFF)")
AGGREGATE_CUSTOMERS = {"SUBTOTAL", "TOTAL"}

promo_parts = []
for sheet in sheet_order:
    df = frames[sheet]
    cols = {c.strip(): c for c in df.columns if isinstance(c, str)}
    pn = cols.get("Product Name")
    inum = cols.get("Item Number")
    promo = next((cols[k] for k in PROMO_COL_VARIANTS if k in cols), None)
    cust = cols.get("Customer")
    if not (pn and inum and promo):
        print(f"  [skip] {sheet}: missing Product Name / Item Number / Promo Discount")
        continue
    keep = [pn, inum, promo] + ([cust] if cust else [])
    sub = df[keep].copy()
    rename = {pn: "Product Name", inum: "Item Number", promo: "Promo Discount"}
    if cust:
        rename[cust] = "Customer"
    sub = sub.rename(columns=rename)
    if "Customer" not in sub.columns:
        sub["Customer"] = "All"

    sub = sub.dropna(subset=["Product Name", "Item Number", "Promo Discount"], how="any")
    sub["Product Name"] = sub["Product Name"].astype(str).str.strip()
    sub["Item Number"] = sub["Item Number"].astype(str).str.strip()
    sub["Customer"] = sub["Customer"].fillna("All").astype(str).str.strip()
    sub = sub[(sub["Product Name"] != "") & (sub["Item Number"] != "")]
    sub["Normalized Name"] = sub["Product Name"].map(normalize_name)
    sub = sub[sub["Normalized Name"] != ""]
    sub = sub[~sub["Customer"].str.upper().isin(AGGREGATE_CUSTOMERS)]
    sub["Promo Discount"] = pd.to_numeric(sub["Promo Discount"], errors="coerce")
    sub = sub.dropna(subset=["Promo Discount"])
    sub = sub[sub["Promo Discount"] > 0]
    if sub.empty:
        continue
    sub = sub.drop_duplicates(subset=["Normalized Name", "Customer", "Promo Discount"])
    sub["sheet"] = sheet
    sub["period_start"] = sheet_periods[sheet][0]
    sub["period_end"] = sheet_periods[sheet][1]
    promo_parts.append(sub)

promo_long = pd.concat(promo_parts, ignore_index=True)
print(f"{len(promo_long)} (normalized product, customer, promo, sheet) observations")
print("customers seen:", sorted(promo_long["Customer"].unique()))
promo_long.head()

3022 (normalized product, customer, promo, sheet) observations
customers seen: ['All', 'Heybox', 'Sonkwo']


,Product Name,Item Number,Promo Discount,Customer,Normalized Name,sheet,period_start,period_end
0,SPY×ANYA: Operation Memories,E05697,0.3000,All,SPYANYAOPERATIONMEMORIES,11月,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,E05367,0.6835,All,HACKGULASTRECODE,11月,2024-11-01,2024-11-30
2,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6800,All,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,11月,2024-11-01,2024-11-30
3,Accel World VS. Sword Art Online Deluxe Edition,E02452,0.6835,All,ACCELWORLDVSSWORDARTONLINEDELUXEEDITION,11月,2024-11-01,2024-11-30
4,Ace Combat 7: Skies Unknown,E03174,0.6716,All,ACECOMBAT7SKIESUNKNOWN,11月,2024-11-01,2024-11-30


In [10]:
promo_long["_idx"] = promo_long["sheet"].map(sheet_idx)
promo_long = promo_long.sort_values(
    ["Normalized Name", "Customer", "Promo Discount", "_idx"]
).reset_index(drop=True)

promo_runs = []
group_keys = ["Normalized Name", "Customer", "Promo Discount"]
for (norm, customer, promo), grp in promo_long.groupby(group_keys, sort=False):
    grp = grp.sort_values("_idx")
    run_start = run_end = None
    prev_idx = None
    for _, row in grp.iterrows():
        if prev_idx is None:
            run_start = row["period_start"]
            run_end = row["period_end"]
            prev_idx = row["_idx"]
            continue
        if row["_idx"] == prev_idx + 1:
            run_end = row["period_end"]
        else:
            promo_runs.append((norm, customer, promo, run_start, run_end))
            run_start = row["period_start"]
            run_end = row["period_end"]
        prev_idx = row["_idx"]
    if prev_idx is not None:
        promo_runs.append((norm, customer, promo, run_start, run_end))

promo_history = pd.DataFrame(
    promo_runs,
    columns=["Normalized Name", "Customer", "Promo Discount", "start_date", "end_date"],
)
promo_history.insert(0, "Product Name", promo_history["Normalized Name"].map(display_names))
promo_history["start_month"] = promo_history["start_date"].dt.strftime("%Y-%m")
promo_history["end_month"] = promo_history["end_date"].dt.strftime("%Y-%m")
promo_history = promo_history[
    ["Product Name", "Normalized Name", "Customer", "Promo Discount",
     "start_month", "end_month", "start_date", "end_date"]
].sort_values(["Product Name", "Customer", "start_date", "Promo Discount"]).reset_index(drop=True)

print(f"{len(promo_history)} promo runs")
print(f"  across {promo_history['Normalized Name'].nunique()} products")
print(f"  across {promo_history[['Normalized Name','Customer']].drop_duplicates().shape[0]} (product, customer) pairs")

out = Path("product_promo_history.csv")
promo_history.drop(columns=["start_date", "end_date"]).to_csv(out, index=False)
print(f"Wrote {out.resolve()}")
promo_history.head(20)

1260 promo runs
  across 179 products
  across 459 (product, customer) pairs
Wrote /Users/christopherdrews/dev/bandai-data-analysis/product_promo_history.csv


,Product Name,Normalized Name,Customer,Promo Discount,start_month,end_month,start_date,end_date
0,.hack//G.U. Last Recode,HACKGULASTRECODE,All,0.6835,2024-11,2024-11,2024-11-01,2024-11-30
1,.hack//G.U. Last Recode,HACKGULASTRECODE,All,0.9000,2024-12,2025-03,2024-12-01,2025-03-31
2,.hack//G.U. Last Recode,HACKGULASTRECODE,All,0.8000,2025-04,2025-04,2025-04-01,2025-04-30
3,.hack//G.U. Last Recode,HACKGULASTRECODE,All,0.9000,2025-06,2025-06,2025-06-01,2025-06-30
4,.hack//G.U. Last Recode,HACKGULASTRECODE,Heybox,0.9000,2025-07,2025-09,2025-07-01,2025-09-30
5,.hack//G.U. Last Recode,HACKGULASTRECODE,Heybox,0.8000,2025-10,2026-04,2025-10-01,2026-04-30
6,.hack//G.U. Last Recode,HACKGULASTRECODE,Heybox,0.9000,2026-04,2026-04,2026-04-01,2026-04-30
7,.hack//G.U. Last Recode,HACKGULASTRECODE,Sonkwo,0.9000,2025-07,2025-07,2025-07-01,2025-07-31
8,.hack//G.U. Last Recode,HACKGULASTRECODE,Sonkwo,0.8000,2025-11,2026-02,2025-11-01,2026-02-28
9,ACE COMBAT 7: SKIES UNKNOWN,ACECOMBAT7SKIESUNKNOWN,All,0.6716,2024-11,2024-11,2024-11-01,2024-11-30


## Task 4 — Resolve Playasia PAX codes by name

`lookup_pax_codes.py` queries the Playasia `/products/filtered` endpoint (type=`digital codes`, digital=1) for each unique Product Name and writes `distinct_products_with_pax.csv` with the top match. This cell loads that output and surfaces rows that need attention: ones with no result, ones where the CSV's existing Item Number differs from the Playasia PAX code, and the original `0` Item Number rows.

In [11]:
pax = pd.read_csv('distinct_products_with_pax.csv', dtype=str).fillna('')
pax['lookup_score'] = pd.to_numeric(pax['lookup_score'], errors='coerce')
print('match_status counts:')
print(pax['match_status'].value_counts().to_string())
print(f"\nmean lookup_score: {pax['lookup_score'].mean():.1f}")
pax.head()

match_status counts:
match_status
differs        177
not_found       63
csv_invalid      2

mean lookup_score: 51.6


,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_total_count,lookup_score,match_status
0,.hack//G.U. Last Recode,E05367,PAX0011620588,.Hack//G.U. Last Recode,1,95.7,differs
1,ACE COMBAT 7: SKIES UNKNOWN,E03174,PAX0012109542,Ace Combat 7: Skies Unknown,7,37.0,differs
2,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05445,PAX0012537704,Ace Combat 7: Skies Unknown - Top Gun: Maveric...,1,61.8,differs
3,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444,,,0,0.0,not_found
4,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05445,PAX0012537704,Ace Combat 7: Skies Unknown - Top Gun: Maveric...,1,61.3,differs


### Rows with no Playasia match

In [12]:
pax[pax['match_status'] == 'not_found'][['Product Name', 'Item Number']]

,Product Name,Item Number
3,ACE COMBAT 7: SKIES UNKNOWN - TOP GUN: Maveric...,E05444
5,ACE COMBAT™ 7: SKIES UNKNOWN - TOP GUN: Maveri...,E05444
8,ARMORED CORE VI FIRES OF RUBICON Standard Edition,E05837
37,DEATH NOTE Killer Within,E06643
38,DEATH NOTE Killer Within Special Edition,E06646
...,...,...
235,That Time I Got Reincarnated as a Slime ISEKAI...,E06534
236,That Time I Got Reincarnated as a Slime ISEKAI...,E06534
238,Towa and the Guardians of the Sacred Tree Delu...,E06936
239,Towa and the Guardians of the Sacred Tree Delu...,L00117


### Rows where the CSV's Item Number was missing (`0`) — recovered PAX codes

In [13]:
pax[pax['match_status'] == 'csv_invalid'][['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score']]

,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_score
72,Digimon Story Time Stranger Deluxe Edition,0,PAX0014928822,Digimon Story Time Stranger (Deluxe Edition),97.7
74,Digimon Story Time Stranger Ultimate Edition,0,PAX0014928856,Digimon Story Time Stranger (Ultimate Edition),97.8


### Lowest-confidence "differs" rows (sorted by fuzzy score — review these manually)

In [14]:
(pax[pax['match_status'] == 'differs']
 .sort_values('lookup_score')
 [['Product Name', 'Item Number', 'lookup_pax_code', 'lookup_label', 'lookup_score', 'lookup_total_count']]
 .head(30))

,Product Name,Item Number,lookup_pax_code,lookup_label,lookup_score,lookup_total_count
154,SCARLET NEXUS,E03557,PAX0011642246,Scarlet Nexus (Deluxe Edition),24.5,3
43,DORAEMON STORY OF SEASONS,E06163,PAX0012561776,Doraemon Story of Seasons: Friends of the Grea...,24.5,2
153,SCARLET NEXUS,E02985,PAX0011642246,Scarlet Nexus (Deluxe Edition),24.5,3
42,DORAEMON STORY OF SEASONS,E03496,PAX0012561776,Doraemon Story of Seasons: Friends of the Grea...,24.5,2
102,MOBILE SUIT GUNDAM SEED BATTLE DESTINY REMASTERED,E06942,PAX0014731554,Mobile Suit Gundam Seed Battle Destiny Remastered,26.5,1
36,DARK SOULS™: REMASTERED,E02716,PAX0010382852,Dark Souls: Remastered,26.7,5
39,DIGIMON STORY TIME STRANGER,E03053,PAX0014928856,Digimon Story Time Stranger (Ultimate Edition),26.8,5
31,DARK SOULS: REMASTERED,E02716,PAX0010382852,Dark Souls: Remastered,27.3,5
109,NARUTO TO BORUTO: SHINOBI STRIKER,E01572,PAX0009688844,Naruto to Boruto: Shinobi Striker,27.3,12
110,NARUTO TO BORUTO: SHINOBI STRIKER,E01927,PAX0009688844,Naruto to Boruto: Shinobi Striker,27.3,12
